# CARLA Python API — Labs 2 · 3 · 4 (notebook unificato)

Questo notebook raccoglie in un unico documento i contenuti dei laboratori 2, 3 e 4 del corso **Smart Vehicular Systems** (UniBo).

---

## Indice

| Sezione | Argomento |
|---|---|
| [Setup](#setup) | Import e connessione al server |
| [Lab 2 — §1](#lab2-connect) | Connessione e timeout |
| [Lab 2 — §2](#lab2-config) | Mappa, meteo, sync/async |
| [Lab 2 — §3](#lab2-actors) | Blueprint e spawn attori |
| [Lab 3 — §1](#lab3-helpers) | Helper functions |
| [Lab 3 — §2](#lab3-spectator) | Spectator + autopilot |
| [Lab 3 — §3](#lab3-waypoints) | Waypoint e debug drawing |
| [Lab 3 — §4](#lab3-control) | Riposizionamento vs controllo |
| [Lab 3 — §5](#lab3-exercises) | Esercizi in classe |
| [Lab 4 — §1](#lab4-camera) | RGB Camera |
| [Lab 4 — §2](#lab4-lidar) | LiDAR |
| [Lab 4 — §3](#lab4-exercises) | Esercizi (template + soluzioni) |

---
**Documentazione ufficiale:** https://carla.readthedocs.io/en/latest/

<a id='setup'></a>
## Setup globale

Esegui questa cella una volta sola prima di qualsiasi sezione.

In [ ]:
import carla
import time
import random
import threading
import queue

try:
    import pygame
except ImportError:
    pygame = None
    print("pygame non disponibile (necessario solo per Lab 3)")

try:
    import cv2
    import numpy as np
except ImportError:
    cv2 = None
    np = None
    print("cv2/numpy non disponibili (necessari solo per Lab 4)")

In [ ]:
client = carla.Client("localhost", 2000)
client.set_timeout(15.0)
world = client.get_world()
spectator = world.get_spectator()

print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

---
<a id='lab2-connect'></a>
# Lab 2 — Connessione, configurazione e attori

**Obiettivi:**
- costruire una connessione stabile client-server;
- configurare mappa, meteo e modalità di esecuzione;
- aggiungere e gestire attori nella scena.

Ref.: [`carla.Client`](https://carla.readthedocs.io/en/latest/python_api/#carla.Client)

> `Client` è un proxy RPC: ogni chiamata API invia una richiesta al server e riceve una risposta.

<a id='lab2-config'></a>
## §2 — Mappa e meteo

Ref.: [`get_available_maps`](https://carla.readthedocs.io/en/latest/python_api/#carla.Client.get_available_maps) · [`load_world`](https://carla.readthedocs.io/en/latest/python_api/#carla.Client.load_world) · [`WeatherParameters`](https://carla.readthedocs.io/en/latest/python_api/#carla.WeatherParameters)

In [ ]:
# Mappe disponibili
client.get_available_maps()

In [ ]:
# Carica Town10HD_Opt (buon dettaglio, costo computazionale contenuto)
client.load_world("Town10HD_Opt")
world = client.get_world()
print("Mappa caricata:", world.get_map().name)

In [ ]:
# Sequenza di presets meteo
weathers = [
    carla.WeatherParameters.CloudyNoon,
    carla.WeatherParameters.MidRainSunset,
    carla.WeatherParameters.HardRainNoon,
    carla.WeatherParameters.ClearSunset,
    carla.WeatherParameters.SoftRainNight,
    carla.WeatherParameters.Default,
]
for weather in weathers:
    world.set_weather(weather)
    print(f"Weather: {weather}")
    time.sleep(3)

### Sync vs Async

| Modalità | Avanzamento | Uso tipico |
|---|---|---|
| **Sync** | solo su `world.tick()` | esperimenti riproducibili, sensori, raccolta dati |
| **Async** | automatico alla velocità massima | demo in tempo reale |

Ref.: [Synchrony and timestep](https://carla.readthedocs.io/en/latest/adv_synchrony_timestep/) · [`WorldSettings`](https://carla.readthedocs.io/en/latest/python_api/#carla.WorldSettings)

In [ ]:
# Attiva modalità sincrona
settings = world.get_settings()
settings.synchronous_mode = True
world.apply_settings(settings)
print("Modalità sincrona attivata")

for i in range(100):
    world.tick()
    print(f"Tick {i+1}/100", end="\r")

# Ripristina modalità asincrona
settings.synchronous_mode = False
world.apply_settings(settings)
client.reload_world()
world = client.get_world()
spectator = world.get_spectator()
print("\nModalità asincrona ripristinata")

<a id='lab2-actors'></a>
## §3 — Blueprint e spawn attori

Ref.: [`get_blueprint_library`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.get_blueprint_library) · [`try_spawn_actor`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.try_spawn_actor)

In [ ]:
# Esplora spawn points
spawn_points = world.get_map().get_spawn_points()
print(f"Spawn points disponibili: {len(spawn_points)}")
for sp in spawn_points[:5]:
    print(sp.location)

In [ ]:
# Filtra blueprint veicoli
bp_lib = world.get_blueprint_library()
vehicle_bps = [bp for bp in bp_lib if 'vehicle' in bp.id]
print(f"Blueprint veicoli: {len(vehicle_bps)}")
for bp in vehicle_bps[:10]:
    print(f"  {bp.id}")

In [ ]:
# Spawn, autopilot e destroy
vehicle_bp = random.choice(vehicle_bps)
vehicle = world.try_spawn_actor(vehicle_bp, spawn_points[0])

if vehicle is None:
    print("Spawn fallito: punto occupato o non valido.")
else:
    print(f"Spawned: {vehicle.type_id}")
    vehicle.set_autopilot(True)
    time.sleep(3)
    vehicle.destroy()
    vehicle = None
    print("Distrutto.")

In [ ]:
# Esercizio Lab2: 50 spawn casuali — conta successi e fallimenti
vehicle_bp = random.choice(vehicle_bps)
success, fails = 0, 0

for _ in range(50):
    sp = random.choice(spawn_points)
    v = world.try_spawn_actor(vehicle_bp, sp)
    if v:
        success += 1
        time.sleep(0.05)
        v.destroy()
    else:
        fails += 1

print(f"Successi: {success} | Fallimenti: {fails}")

---
<a id='lab3-helpers'></a>
# Lab 3 — Spectator, Waypoint e Controllo

**Obiettivi:**
- riutilizzare helper per esperimenti veloci;
- seguire veicoli con la camera spectator;
- ispezionare waypoint e topologia stradale con debug drawing;
- confrontare riposizionamento (`set_transform`) vs controllo (`apply_control`).

Ref.: [Core actors](https://carla.readthedocs.io/en/latest/core_actors/) · [Map & waypoints](https://carla.readthedocs.io/en/latest/core_map/) · [DebugHelper](https://carla.readthedocs.io/en/latest/python_api/#carladebughelper)

<a id='lab3-helpers'></a>
## §1 — Helper functions

In [ ]:
def move_spectator_to(transform, spectator, distance=7.0, x=0, y=0, z=3.0,
                      yaw=0, pitch=-15, roll=0):
    """Posiziona lo spectator dietro/sopra il transform fornito."""
    back_location = transform.location - transform.get_forward_vector() * distance
    back_location.x += x
    back_location.y += y
    back_location.z += z
    transform.rotation.yaw   += yaw
    transform.rotation.pitch  = pitch
    transform.rotation.roll   = roll
    spectator.set_transform(carla.Transform(back_location, transform.rotation))


def spawn_vehicle_simple(world, vehicle_index=0, spawn_index=0):
    """Spawn base per Lab 3 (indice blueprint + indice spawn)."""
    bp = world.get_blueprint_library().filter('vehicle.*')[vehicle_index]
    sp = world.get_map().get_spawn_points()[spawn_index]
    return world.spawn_actor(bp, sp)


def draw_on_screen(world, transform, content="O",
                   color=carla.Color(0, 255, 0), life_time=20):
    world.debug.draw_string(transform.location, content,
                            color=color, life_time=life_time)

<a id='lab3-spectator'></a>
## §2 — Spectator + autopilot

In [ ]:
vehicle = spawn_vehicle_simple(world)
vehicle.set_autopilot(True)
move_spectator_to(vehicle.get_transform(), spectator)

try:
    for _ in range(100):
        world.tick()
        time.sleep(0.1)
finally:
    vehicle.destroy()

<a id='lab3-waypoints'></a>
## §3 — Waypoint e debug drawing

Ref.: [`get_waypoint`](https://carla.readthedocs.io/en/latest/python_api/#carla.Map.get_waypoint) · [`get_topology`](https://carla.readthedocs.io/en/latest/python_api/#carla.Map.get_topology)

In [ ]:
# Campiona waypoint durante la guida
vehicle = spawn_vehicle_simple(world)
vehicle.set_autopilot(True)
move_spectator_to(vehicle.get_transform(), spectator)

waypoints_list = []
for _ in range(10):
    wp = world.get_map().get_waypoint(vehicle.get_transform().location)
    waypoints_list.append(wp)
    world.tick()
    time.sleep(1)

vehicle.destroy()
for wp in waypoints_list:
    print(wp.transform.location)

In [ ]:
# Disegna waypoint campionati
for wp in waypoints_list:
    world.debug.draw_string(
        wp.transform.location, 'WP',
        color=carla.Color(r=0, g=255, b=0), life_time=20.0
    )

In [ ]:
# Topologia stradale: percorri un segmento
roads = world.get_map().get_topology()
waypoint = roads[0][0]
draw_on_screen(world, waypoint.transform)

for _ in range(1000):
    waypoint = waypoint.next(1)[0]
    draw_on_screen(world, waypoint.transform)
    time.sleep(0.01)

In [ ]:
# Colora segmenti stradali con colori diversi
def color_generator():
    colors = [carla.Color(r=r, g=g, b=b)
              for r in [0, 255] for g in [0, 255] for b in [0, 255]]
    while True:
        for c in colors:
            yield c

for i, color in zip(range(len(roads)), color_generator()):
    wp = roads[i][0]
    for _ in range(1000):
        wp = wp.next(3)[0]
        draw_on_screen(world, wp.transform, color=color, life_time=10)
    time.sleep(0.5)

<a id='lab3-control'></a>
## §4 — Riposizionamento vs controllo

| Operazione | Meccanismo | Uso |
|---|---|---|
| `set_transform` | teleport istantaneo | setup / debug |
| `apply_control` | moto fisico | guida reale |

In [ ]:
# Movimento simulato via set_transform (teleport)
vehicle = spawn_vehicle_simple(world)
roads = world.get_map().get_topology()
waypoint = roads[0][0]
vehicle.set_transform(waypoint.transform)
move_spectator_to(waypoint.transform, spectator)

try:
    for _ in range(50):
        waypoint = waypoint.next(1)[0]
        vehicle.set_transform(waypoint.transform)
        time.sleep(0.2)
except KeyboardInterrupt:
    pass
finally:
    vehicle.destroy()

In [ ]:
# Controllo reale con VehicleControl
vehicle = spawn_vehicle_simple(world)
move_spectator_to(vehicle.get_transform(), spectator)
throttle = 0.6

def move_forward(vehicle, duration):
    ctrl = carla.VehicleControl(throttle=throttle)
    vehicle.apply_control(ctrl)
    t0 = time.time()
    while time.time() - t0 < duration:
        world.tick(); time.sleep(0.1)

def steer_left(vehicle):
    ctrl = carla.VehicleControl(throttle=throttle, steer=-0.38)
    vehicle.apply_control(ctrl)
    world.tick(); time.sleep(2)

try:
    move_forward(vehicle, 5)
    steer_left(vehicle)
    move_forward(vehicle, 10)
finally:
    vehicle.destroy()

In [ ]:
# Controllo casuale (throttle + steer random)
vehicle = spawn_vehicle_simple(world)
move_spectator_to(vehicle.get_transform(), spectator)

try:
    for _ in range(200):
        ctrl = carla.VehicleControl(
            throttle=random.uniform(0.5, 1.0),
            steer=random.uniform(-1.0, 1.0)
        )
        vehicle.apply_control(ctrl)
        world.tick(); time.sleep(0.1)
except KeyboardInterrupt:
    pass
finally:
    vehicle.destroy()

<a id='lab3-exercises'></a>
## §5 — Esercizi Lab 3

### Esercizio 1: Multi-view spectator
Tre viste selezionabili prima dell'avvio: chase (`1`), top (`2`), side (`3`).

> Richiede pygame.

In [ ]:
if pygame is None:
    raise ImportError("pygame non installato")

vehicle = spawn_vehicle_simple(world)
vehicle.set_autopilot(True)

pygame.init()
pygame.display.set_mode((420, 240))
camera_mode = 1  # 1=chase, 2=top, 3=side

if camera_mode == 1:
    move_spectator_to(vehicle.get_transform(), spectator, distance=7.0, z=3.0, pitch=-15)
elif camera_mode == 2:
    move_spectator_to(vehicle.get_transform(), spectator, distance=0.0, z=35.0, pitch=-90)
elif camera_mode == 3:
    move_spectator_to(vehicle.get_transform(), spectator, distance=0.0, y=8.0, z=3.0, yaw=90, pitch=-15)

running = True
try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False
        world.tick(); time.sleep(0.03)
finally:
    vehicle.destroy()
    pygame.quit()

### Esercizio 2: Safety ring

In [ ]:
vehicle = spawn_vehicle_simple(world)
vehicle.set_autopilot(True)

ring_radius = 3.0
ring_height = 0.6
offsets = [
    carla.Location(x= ring_radius, y= 0,           z=ring_height),
    carla.Location(x=-ring_radius, y= 0,           z=ring_height),
    carla.Location(x= 0,          y= ring_radius,  z=ring_height),
    carla.Location(x= 0,          y=-ring_radius,  z=ring_height),
    carla.Location(x= ring_radius, y= ring_radius, z=ring_height),
    carla.Location(x= ring_radius, y=-ring_radius, z=ring_height),
    carla.Location(x=-ring_radius, y= ring_radius, z=ring_height),
    carla.Location(x=-ring_radius, y=-ring_radius, z=ring_height),
]
move_spectator_to(vehicle.get_transform(), spectator, distance=0.0, y=8.0, z=3.0, yaw=90, pitch=-15)

try:
    for _ in range(300):
        center = vehicle.get_transform().location
        for off in offsets:
            world.debug.draw_point(center + off, size=0.1,
                                   color=carla.Color(255, 0, 0), life_time=0.1)
        world.tick(); time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    vehicle.destroy()

### Esercizio 3: Guida manuale progressiva

> Richiede pygame. Tasti: `W` accelera · `S` frena · `A`/`D` sterza · `ESC` esci.

In [ ]:
if pygame is None:
    raise ImportError("pygame non installato")

vehicle = spawn_vehicle_simple(world)
pygame.init()
pygame.display.set_mode((420, 240))

control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=0.0)
THROTTLE_STEP, STEER_STEP = 0.03, 0.04
running = True

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

move_spectator_to(vehicle.get_transform(), spectator, distance=0.0, z=35.0, pitch=-90)

try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False

        keys = pygame.key.get_pressed()
        control.throttle = clamp(control.throttle + (THROTTLE_STEP if keys[pygame.K_w] else -THROTTLE_STEP), 0.0, 1.0)
        control.brake    = clamp(control.brake    + (THROTTLE_STEP if keys[pygame.K_s] else -THROTTLE_STEP), 0.0, 1.0)
        if keys[pygame.K_a]:
            control.steer = clamp(control.steer - STEER_STEP, -1.0, 1.0)
        elif keys[pygame.K_d]:
            control.steer = clamp(control.steer + STEER_STEP, -1.0, 1.0)
        else:
            control.steer = 0.0

        vehicle.apply_control(control)
        world.tick(); time.sleep(0.03)
finally:
    vehicle.destroy()
    pygame.quit()

---
<a id='lab4-camera'></a>
# Lab 4 — RGB Camera e LiDAR

**Obiettivi:** montare sensori su un veicolo, elaborare i dati in callback, costruire logiche di controllo basate su percezione.

Ref.: [Sensors reference](https://carla.readthedocs.io/en/latest/ref_sensors/)

## Helper functions Lab 4

In [ ]:
def move_spectator_to_v2(transform, spectator, distance=7.0, z=3.0, pitch=-15.0):
    """Versione semplificata per Lab 4 (senza offset laterali)."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))


def spawn_vehicle_v2(world, spawn_index=0, vehicle_filter="vehicle.tesla.model3", autopilot=False):
    points = world.get_map().get_spawn_points()
    bps = world.get_blueprint_library().filter(vehicle_filter) or \
          world.get_blueprint_library().filter("vehicle.*")
    for k in range(len(points)):
        actor = world.try_spawn_actor(random.choice(bps),
                                      points[(spawn_index + k) % len(points)])
        if actor is not None:
            actor.set_autopilot(autopilot)
            return actor
    raise RuntimeError("Could not spawn vehicle")


def spawn_vehicle_ahead(world, ref_vehicle, distances=(20.0, 28.0, 36.0),
                        same_lane_first=True, retries=6):
    """Spawna un veicolo davanti al veicolo di riferimento."""
    ego_tf  = ref_vehicle.get_transform()
    ego_loc = ego_tf.location
    fwd     = ego_tf.get_forward_vector()
    bps     = world.get_blueprint_library().filter("vehicle.*")

    # Phase 1: offset diretto
    direct = sorted(set(float(d) for d in distances))
    direct += [d + 4.0 for d in direct]
    for d in direct:
        loc = ego_loc + fwd * d
        loc.z += 0.30
        actor = world.try_spawn_actor(random.choice(bps),
                                      carla.Transform(loc, ego_tf.rotation))
        if actor is not None:
            actor.set_autopilot(False)
            return actor

    # Phase 2: waypoint davanti
    road_map = world.get_map()
    ego_wp   = road_map.get_waypoint(ego_loc, project_to_road=True,
                                     lane_type=carla.LaneType.Driving)
    wp_candidates = []
    d_pool = sorted(set(float(d) for d in distances) |
                    set(float(d) + 8.0 for d in distances))
    for d in d_pool:
        try:
            cands = ego_wp.next(float(d))
        except RuntimeError:
            cands = []
        if same_lane_first:
            cands = sorted(cands, key=lambda w: (
                w.road_id != ego_wp.road_id,
                w.lane_id != ego_wp.lane_id,
                abs(w.lane_id - ego_wp.lane_id)))
        wp_candidates.extend(cands)

    for _ in range(max(1, int(retries))):
        for wp in wp_candidates:
            actor = world.try_spawn_actor(random.choice(bps), wp.transform)
            if actor is not None:
                actor.set_autopilot(False)
                return actor
        world.tick(); time.sleep(0.02)

    raise RuntimeError("Could not spawn or find a target vehicle ahead")


def spawn_camera(world, attach_to, transform, width=640, height=360, fov=95, tick=0.05):
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov",          str(fov))
    bp.set_attribute("sensor_tick",  str(tick))
    return world.spawn_actor(bp, transform, attach_to=attach_to)


def spawn_lidar(world, attach_to, transform,
                channels=32, points_per_second=56000,
                rotation_frequency=20, range_m=35):
    bp = world.get_blueprint_library().find("sensor.lidar.ray_cast")
    bp.set_attribute("channels",           str(channels))
    bp.set_attribute("points_per_second",  str(points_per_second))
    bp.set_attribute("rotation_frequency", str(rotation_frequency))
    bp.set_attribute("range",              str(range_m))
    return world.spawn_actor(bp, transform, attach_to=attach_to)


def image_to_bgr(image):
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()


def lidar_to_numpy(measurement):
    pts = np.frombuffer(measurement.raw_data, dtype=np.float32)
    return np.reshape(pts, (-1, 4))


def safe_destroy(actors):
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

## §1 — RGB Camera

### Parametri principali

| Parametro | Attributo blueprint | Effetto |
|---|---|---|
| `width`, `height` | `image_size_x/y` | Risoluzione |
| `fov` | `fov` | Campo visivo orizzontale (gradi) |
| `tick` | `sensor_tick` | Secondi tra frame (`0.05` ≈ 20 FPS) |

Pipeline dati: `carla.Image` → `raw_data` (BGRA bytes) → `image_to_bgr` → OpenCV BGR.

In [ ]:
# Stampa tutti gli attributi della camera RGB
bp = world.get_blueprint_library().find("sensor.camera.rgb")
print("Camera blueprint id:", bp.id)

def _attr_value(attr):
    for name in ("default_value", "value", "default"):
        if hasattr(attr, name):
            try:
                v = getattr(attr, name)
                if v is not None: return str(v)
            except Exception: pass
    try: return str(attr)
    except Exception: return "<n/a>"

for attr in bp:
    rec = ", ".join(str(v) for v in list(getattr(attr, "recommended_values", []))[:8])
    print(f"{attr.id:26} value={_attr_value(attr):>10} modifiable={attr.is_modifiable} rec=[{rec}]")

In [ ]:
# Demo — Camera RGB su veicolo in movimento (premi 'q' per uscire)
vehicle = camera = None
latest  = {"frame": None}

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=12, autopilot=True)
    cam_tf  = carla.Transform(carla.Location(x=1.8, z=1.4), carla.Rotation(pitch=-6.0))
    camera  = spawn_camera(world, vehicle, cam_tf, width=800, height=450, fov=95, tick=0.05)
    camera.listen(lambda img: latest.update({"frame": image_to_bgr(img)}))

    t0 = time.time()
    while time.time() - t0 < 20.0:
        if latest["frame"] is not None:
            cv2.imshow("RGB Camera", latest["frame"])
        move_spectator_to_v2(vehicle.get_transform(), spectator, distance=8.0, z=3.0)
        if cv2.waitKey(1) & 0xFF == ord("q"): break
        world.tick(); time.sleep(0.02)
finally:
    if camera: camera.stop()
    safe_destroy([camera, vehicle])
    cv2.destroyAllWindows()

In [ ]:
# Demo — Confronto FOV 60 (sinistra) vs FOV 110 (destra)
vehicle  = None
cameras  = []
fqs      = {"fov60": queue.Queue(1), "fov110": queue.Queue(1)}
latest   = {k: np.zeros((360, 640, 3), dtype=np.uint8) for k in fqs}

def make_fov_cb(name):
    q = fqs[name]
    def _cb(img):
        frame = image_to_bgr(img)
        if q.full():
            try: q.get_nowait()
            except queue.Empty: pass
        try: q.put_nowait(frame)
        except queue.Full: pass
    return _cb

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=14, autopilot=True)
    cam_tf  = carla.Transform(carla.Location(x=1.8, z=1.4), carla.Rotation(pitch=-6.0))
    for name, fov in [("fov60", 60), ("fov110", 110)]:
        cam = spawn_camera(world, vehicle, cam_tf, fov=fov)
        cam.listen(make_fov_cb(name))
        cameras.append(cam)

    t0 = time.time()
    while time.time() - t0 < 20.0:
        for name, q in fqs.items():
            try: latest[name] = q.get_nowait()
            except queue.Empty: pass
        left  = latest["fov60"].copy()
        right = latest["fov110"].copy()
        cv2.putText(left,  "FOV 60",  (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)
        cv2.putText(right, "FOV 110", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)
        cv2.imshow("FOV comparison", np.hstack([left, right]))
        move_spectator_to_v2(vehicle.get_transform(), spectator, distance=8.0, z=3.0)
        if cv2.waitKey(1) & 0xFF == ord("q"): break
        world.tick(); time.sleep(0.02)
finally:
    for c in cameras: c.stop()
    safe_destroy(cameras + [vehicle])
    cv2.destroyAllWindows()

<a id='lab4-lidar'></a>
## §2 — LiDAR

### Parametri principali

| Parametro | Effetto |
|---|---|
| `channels` | Layer verticali |
| `points_per_second` | Densità nuvola |
| `rotation_frequency` | Update rate (Hz) |
| `range_m` | Distanza massima (m) |

Formato dati: `raw_data` → `float32` array → reshape `(-1, 4)` → colonne `[x, y, z, intensity]`.

In [ ]:
# Stampa attributi blueprint LiDAR
bp = world.get_blueprint_library().find("sensor.lidar.ray_cast")
print("LiDAR blueprint id:", bp.id)
for attr in bp:
    rec = ", ".join(str(v) for v in list(getattr(attr, "recommended_values", []))[:8])
    print(f"{attr.id:26} value={_attr_value(attr):>10} modifiable={attr.is_modifiable} rec=[{rec}]")

In [ ]:
# Demo — LiDAR top-view + smooth braking (premi 'q' per uscire)
vehicle = target = lidar = None
state   = {"d_front_raw": None, "d_front": None, "pcd_img": None, "pcd_prev": None}

canvas_h, canvas_w = 760, 760
scale               = 16.0
origin_x, origin_y  = canvas_w // 2, canvas_h - 42

def make_canvas():
    img = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    cv2.line(img, (origin_x, origin_y), (origin_x, 20),             (45,45,45), 1)
    cv2.line(img, (35, origin_y), (canvas_w-35, origin_y),          (45,45,45), 1)
    cv2.putText(img, "ego",       (origin_x+6, origin_y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)
    cv2.putText(img, "x forward", (origin_x+10, 24),        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)
    return img

def on_lidar(measurement):
    pts = lidar_to_numpy(measurement)
    x, y, z = pts[:,0], pts[:,1], pts[:,2]
    vis   = (x>-6.0)&(x<40.0)&(np.abs(y)<18.0)&(z>-2.5)&(z<2.8)
    front = (x>0.8) &(x<32.0)&(np.abs(y)<2.0)  &(z>-1.7)&(z<2.2)

    if np.any(front):
        d   = np.sqrt(x[front]**2 + y[front]**2)
        raw = float(np.percentile(d, 10))
        state["d_front_raw"] = raw
        prev = state["d_front"]
        state["d_front"] = raw if prev is None else (0.82*prev + 0.18*raw)
    else:
        state["d_front_raw"] = state["d_front"] = None

    img = make_canvas()
    if vis.any():
        px = (origin_x + y[vis]*scale).astype(np.int32)
        py = (origin_y - x[vis]*scale).astype(np.int32)
        ok = (px>=0)&(px<canvas_w)&(py>=0)&(py<canvas_h)
        img[py[ok], px[ok]] = (175,175,175)
    if front.any():
        pxf = (origin_x + y[front]*scale).astype(np.int32)
        pyf = (origin_y - x[front]*scale).astype(np.int32)
        okf = (pxf>=0)&(pxf<canvas_w)&(pyf>=0)&(pyf<canvas_h)
        img[pyf[okf], pxf[okf]] = (0,220,255)
        df  = np.sqrt(x[front]**2 + y[front]**2)
        idx = int(np.argmin(df))
        cx  = int(origin_x + y[front][idx]*scale)
        cy  = int(origin_y - x[front][idx]*scale)
        if 0<=cx<canvas_w and 0<=cy<canvas_h:
            cv2.circle(img, (cx,cy), 5, (0,0,255), -1)
    if state["d_front"] is not None:
        cv2.putText(img, "d_est={:.2f} m".format(state["d_front"]),
                    (20,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    prev_img = state["pcd_prev"]
    if prev_img is not None:
        img = cv2.addWeighted(prev_img, 0.35, img, 0.65, 0.0)
    state["pcd_img"] = state["pcd_prev"] = img

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=25, autopilot=False)
    try:
        target = spawn_vehicle_ahead(world, vehicle, distances=(16.,20.,24.), retries=10)
        ego_tf = vehicle.get_transform()
        loc    = ego_tf.location + ego_tf.get_forward_vector()*20.0
        loc.z += 0.30
        target.set_transform(carla.Transform(loc, ego_tf.rotation))
        target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    except RuntimeError as e:
        target = None; print("Warning:", e)

    lidar = spawn_lidar(world, vehicle,
                        carla.Transform(carla.Location(x=1.8, z=2.2)),
                        channels=64, points_per_second=120000,
                        rotation_frequency=30, range_m=50)
    lidar.listen(on_lidar)

    stop_d, slow_d, max_th = 7.5, 16.0, 0.35
    throttle = brake = 0.0
    last_print = t0 = time.time()

    while time.time() - t0 < 26.0:
        d_est = state["d_front"]
        if d_est is None:        tgt_th, tgt_br = max_th, 0.0
        elif d_est <= stop_d:    tgt_th, tgt_br = 0.0, 0.90
        else:
            alpha  = min((d_est - stop_d)/(slow_d - stop_d), 1.0)
            tgt_th = 0.08 + (max_th-0.08)*alpha
            tgt_br = 0.45*(1.0-alpha)
        throttle = 0.84*throttle + 0.16*tgt_th
        brake    = 0.84*brake    + 0.16*tgt_br
        if throttle < 0.02: throttle = 0.0
        if brake    < 0.02: brake    = 0.0
        vehicle.apply_control(carla.VehicleControl(
            throttle=float(np.clip(throttle,0,1)),
            brake   =float(np.clip(brake,   0,1)), steer=0.0))

        d_gt = None
        if target is not None:
            d_gt = vehicle.get_transform().location.distance(
                   target.get_transform().location)
            world.debug.draw_string(
                target.get_transform().location + carla.Location(z=1.8),
                "TARGET", life_time=0.08, color=carla.Color(0,255,255))

        pcd = state["pcd_img"]
        if pcd is not None:
            frame = pcd.copy()
            if d_gt is not None:
                cv2.putText(frame, "d_gt={:.2f} m".format(d_gt),
                            (20,58), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (80,220,80), 2)
            cv2.imshow("LiDAR top-view", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"): break

        if time.time() - last_print > 1.0:
            print("d_est={} d_gt={} th={:.2f} br={:.2f}".format(
                "{:.2f}".format(d_est) if d_est else "None",
                "{:.2f}".format(d_gt)  if d_gt  else "None",
                throttle, brake))
            last_print = time.time()

        move_spectator_to_v2(vehicle.get_transform(), spectator, distance=11.0, z=5.0, pitch=-18.0)
        world.tick(); time.sleep(0.015)
finally:
    if lidar: lidar.stop()
    safe_destroy([lidar, target, vehicle])
    cv2.destroyAllWindows()

<a id='lab4-exercises'></a>
## §3 — Esercizi Lab 4

---
### Esercizio 1: Streamer cockpit (4 camere)

4 viste sincronizzate: frontale, specchio sinistro, specchio destro, retrovisore.

Ref.: [RGB camera](https://carla.readthedocs.io/en/latest/ref_sensors/#rgb-camera)

In [ ]:
# Template — Esercizio 1
vehicle  = None
cameras  = []
fqs      = {}
lf       = {}
res      = (480, 270)

specs = {
    "front": carla.Transform(carla.Location(x=1.8,  z=1.4),  carla.Rotation(pitch=-6.0)),
    "left":  carla.Transform(carla.Location(x=0.1,  y=-0.75, z=1.3), carla.Rotation(yaw=-145.0)),
    "right": carla.Transform(carla.Location(x=0.1,  y= 0.75, z=1.3), carla.Rotation(yaw= 145.0)),
    "rear":  carla.Transform(carla.Location(x=-2.1, z=1.3),  carla.Rotation(yaw=180.0)),
}

stop_event = threading.Event()
quit_event = threading.Event()

def make_cb(name):
    q = fqs[name]
    def _cb(image):
        frame = image_to_bgr(image)
        if q.full():
            try: q.get_nowait()
            except queue.Empty: pass
        try: q.put_nowait(frame)
        except queue.Full: pass
    return _cb

def viewer_loop():
    for name in specs: cv2.namedWindow("Ex1-"+name, cv2.WINDOW_AUTOSIZE)
    while not stop_event.is_set():
        for name, q in fqs.items():
            try: lf[name] = q.get_nowait()
            except queue.Empty: pass
            if name in lf: cv2.imshow("Ex1-"+name, lf[name])
        if cv2.waitKey(1) & 0xFF == ord("q"):
            quit_event.set(); stop_event.set(); break
    cv2.destroyAllWindows()

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=12, autopilot=True)
    for name, tf in specs.items():
        cam = spawn_camera(world, vehicle, tf, width=res[0], height=res[1])
        cameras.append(cam)
        fqs[name] = queue.Queue(maxsize=1)
        lf[name]  = np.zeros((res[1], res[0], 3), dtype=np.uint8)
        cam.listen(make_cb(name))

    vt = threading.Thread(target=viewer_loop, daemon=True)
    vt.start()

    while not quit_event.is_set():
        move_spectator_to_v2(vehicle.get_transform(), spectator)
        world.tick(); time.sleep(0.02)
finally:
    stop_event.set()
    for cam in cameras: cam.stop()
    safe_destroy(cameras + [vehicle])

#### Soluzione Esercizio 1

In [ ]:
# Soluzione — Esercizio 1 (identica al template; il template era già completo)
# Vedi cella precedente — la soluzione coincide con il template completato sopra.

---
### Esercizio 2: Camera HUD + Edge Vision Dashboard

Due pannelli affiancati: RGB con HUD (velocità + reticolo) | Canny edges.

Ref.: [RGB camera](https://carla.readthedocs.io/en/latest/ref_sensors/#rgb-camera) · [Vehicle velocity](https://carla.readthedocs.io/en/latest/python_api/)

In [ ]:
# Template — Esercizio 2
vehicle = camera = None
state   = {"frame": None}

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=12, autopilot=True)
    camera  = spawn_camera(world, vehicle,
                           carla.Transform(carla.Location(x=1.8, z=1.4), carla.Rotation(pitch=-6.0)),
                           width=960, height=540, fov=95, tick=0.05)

    camera.listen(lambda img: state.update({"frame": image_to_bgr(img)}))

    while True:
        frame = state["frame"]
        if frame is not None:
            # HUD
            hud = frame.copy()
            v     = vehicle.get_velocity()
            speed = 3.6 * np.sqrt(v.x**2 + v.y**2 + v.z**2)
            h, w  = hud.shape[:2]
            cx, cy = w//2, h//2
            cv2.line(hud, (cx-20, cy), (cx+20, cy), (0,255,255), 2)
            cv2.line(hud, (cx, cy-20), (cx, cy+20), (0,255,255), 2)
            cv2.putText(hud, "speed = {:.1f} km/h".format(speed),
                        (20,38), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)
            # Canny
            gray      = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges_bgr = cv2.cvtColor(cv2.Canny(gray, 80, 160), cv2.COLOR_GRAY2BGR)
            cv2.imshow("HUD + Canny", np.hstack([hud, edges_bgr]))
            if cv2.waitKey(1) & 0xFF == ord("q"): break

        move_spectator_to_v2(vehicle.get_transform(), spectator, distance=8.0, z=3.0)
        world.tick(); time.sleep(0.02)
except KeyboardInterrupt:
    pass
finally:
    if camera: camera.stop()
    safe_destroy([camera, vehicle])
    cv2.destroyAllWindows()

---
### Esercizio 3: LiDAR Safety Brake

LiDAR montato sul veicolo ego; stima distanza ostacolo frontale; controllo con soglie.

Corridoio frontale ristretto, conteggio minimo punti, hold-time per evitare flicker.

Ref.: [LiDAR sensor](https://carla.readthedocs.io/en/latest/ref_sensors/#lidar-sensor) · [`VehicleControl`](https://carla.readthedocs.io/en/latest/python_api/#carlavehiclecontrol)

In [ ]:
# Template — Esercizio 3 (placeholder da completare)
vehicle = target = lidar = None
state   = {"d_front": None, "last_seen_ts": 0.0}

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=25, autopilot=False)
    try:
        target = spawn_vehicle_ahead(world, vehicle, distances=(18.,22.,26.), retries=8)
        ego_tf = vehicle.get_transform()
        loc    = ego_tf.location + ego_tf.get_forward_vector()*18.0
        loc.z += 0.30
        target.set_transform(carla.Transform(loc, ego_tf.rotation))
        target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    except RuntimeError as e:
        target = None; print("Warning:", e)

    lidar = spawn_lidar(world, vehicle, carla.Transform(carla.Location(x=1.8, z=2.2)))

    def on_lidar(measurement):
        pts = lidar_to_numpy(measurement)
        x, y, z = pts[:,0], pts[:,1], pts[:,2]
        mask = (x>3.0)&(x<22.0)&(np.abs(y)<1.6)&(z>-1.5)&(z<1.5)
        if np.count_nonzero(mask) >= 12:
            d = np.sqrt(x[mask]**2 + y[mask]**2)
            state["d_front"]      = float(np.percentile(d, 20))
            state["last_seen_ts"] = time.time()
        else:
            if time.time() - state["last_seen_ts"] > 0.25:
                state["d_front"] = None

    lidar.listen(on_lidar)
    last_print = 0.0

    while True:
        d = state["d_front"]
        # TODO: implementa logica throttle/brake in base a d
        # se d è None o > 14 → throttle=0.35, brake=0
        # se d > 9 → throttle=0.12, brake=0.20
        # altrimenti → throttle=0, brake=1.0
        world.tick(); time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    if lidar: lidar.stop()
    safe_destroy([lidar, target, vehicle])

#### Soluzione Esercizio 3

In [ ]:
# Soluzione — Esercizio 3
vehicle = target = lidar = None
state   = {"d_front": None, "last_seen_ts": 0.0}

try:
    vehicle = spawn_vehicle_v2(world, spawn_index=25, autopilot=False)
    try:
        target = spawn_vehicle_ahead(world, vehicle, distances=(18.,22.,26.), retries=8)
        ego_tf = vehicle.get_transform()
        loc    = ego_tf.location + ego_tf.get_forward_vector()*18.0
        loc.z += 0.30
        target.set_transform(carla.Transform(loc, ego_tf.rotation))
        target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    except RuntimeError as e:
        target = None; print("Warning:", e)

    lidar = spawn_lidar(world, vehicle, carla.Transform(carla.Location(x=1.8, z=2.2)))

    def on_lidar(measurement):
        pts = lidar_to_numpy(measurement)
        x, y, z = pts[:,0], pts[:,1], pts[:,2]
        mask = (x>3.0)&(x<22.0)&(np.abs(y)<1.6)&(z>-1.5)&(z<1.5)
        if np.count_nonzero(mask) >= 12:
            d = np.sqrt(x[mask]**2 + y[mask]**2)
            state["d_front"]      = float(np.percentile(d, 20))
            state["last_seen_ts"] = time.time()
        else:
            if time.time() - state["last_seen_ts"] > 0.25:
                state["d_front"] = None

    lidar.listen(on_lidar)
    last_print = 0.0

    while True:
        d = state["d_front"]

        if d is None or d > 14.0:
            ctrl = carla.VehicleControl(throttle=0.35, brake=0.0,  steer=0.0)
        elif d > 9.0:
            ctrl = carla.VehicleControl(throttle=0.12, brake=0.20, steer=0.0)
        else:
            ctrl = carla.VehicleControl(throttle=0.0,  brake=1.0,  steer=0.0)

        vehicle.apply_control(ctrl)

        if d is not None:
            world.debug.draw_string(
                vehicle.get_transform().location + carla.Location(z=2.2),
                "d_front={:.2f} m".format(d),
                life_time=0.08, color=carla.Color(255,255,0))
        if target is not None:
            world.debug.draw_string(
                target.get_transform().location + carla.Location(z=1.8),
                "TARGET", life_time=0.08, color=carla.Color(0,255,255))

        if time.time() - last_print > 1.0:
            print("d_front={}".format("{:.2f} m".format(d) if d else "None"))
            last_print = time.time()

        move_spectator_to_v2(vehicle.get_transform(), spectator, distance=10.0, z=4.0, pitch=-16.0)
        world.tick(); time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    if lidar: lidar.stop()
    safe_destroy([lidar, target, vehicle])